# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The quick brown fox jumps over the lazy dog."
print(sample_sentence)

The quick brown fox jumps over the lazy dog.


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | quick        |  4248
    3 | brown        |  2829
    4 | fox          |  4419
    5 | jumps        | 14523
    6 | over         |  2058
    7 | the          |  1996
    8 | lazy         | 13971
    9 | dog          |  3899
   10 | .            |  1012
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]

### Exercise 1 reflection
- **[CLS] Token**: This token is always inserted at the beginning of the input sequence. In classification tasks, the final hidden state corresponding to this token is often used as the aggregate sequence representation for classification.
- **[SEP] Token**: This token is used to separate different segments of text, or to mark the end of a single sequence. For example, in a question-answering task, it separates the question from the context.
- **[PAD] Token**: This token is added to sequences to make them all the same length (up to `max_length`). Models are often designed to work with fixed-size inputs, so padding ensures uniformity.
- **Attention Mask**: The attention mask is a binary tensor (0s and 1s) that tells the model which tokens to pay attention to and which to ignore. `1` indicates a real token, and `0` indicates a padding token. During self-attention, the model will mask out (or ignore) the positions marked with `0`, preventing padding tokens from influencing the attention weights and computations. This ensures that the model focuses only on the meaningful parts of the input sequence.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "TODO: add a sentence whose sentiment you want to test"
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9761614203453064}]

### Exercise 2 reflection
- **Does the predicted label match your expectation? Why or why not?**
  Yes, the predicted label `POSITIVE` perfectly matches my expectation. The sentence "This movie was an absolute masterpiece; I loved every minute of it!" contains strongly positive sentiment words like "masterpiece" and "loved," making it clearly positive.

- **How confident is the model and what does the score tell you?**
  The model is highly confident with a `score` of approximately `0.9998`. This score represents the probability that the sentence belongs to the `POSITIVE` class. A score very close to 1 (like this one) indicates a very strong conviction by the model that its prediction is correct. Conversely, a score close to 0.5 would indicate uncertainty (as it's a binary classification task).

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {"input_ids": encoding["input_ids"].to(self.device),
                "attention_mask": encoding["attention_mask"].to(self.device)}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).squeeze().tolist()

        # Assuming binary classification where 0 is negative and 1 is positive
        # For distilbert-base-uncased-finetuned-sst-2-english:
        # The model's label mapping is usually available via model.config.id2label
        if self.model.config.id2label[0] == "NEGATIVE":
            label_mapping = {0: "NEGATIVE", 1: "POSITIVE"}
        else: # Defaulting to positive first if not explicitly negative first
            label_mapping = {0: "POSITIVE", 1: "NEGATIVE"}

        predicted_class_id = torch.argmax(logits, dim=1).item()
        predicted_label = label_mapping[predicted_class_id]
        predicted_probability = probabilities[predicted_class_id]

        return {"label": predicted_label, "score": predicted_probability}

In [13]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "This movie was absolutely fantastic! A real gem.",
    "I couldn't stand the acting; it was truly dreadful.",
    "The plot was a bit confusing, but the visuals were stunning."
]
for text in samples:
    print(f"Sentence: '{text}'")
    prediction = analyzer.predict(text)
    print(f"Prediction: {prediction['label']}, Score: {prediction['score']:.4f}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence: 'This movie was absolutely fantastic! A real gem.'
Prediction: POSITIVE, Score: 0.9999

Sentence: 'I couldn't stand the acting; it was truly dreadful.'
Prediction: NEGATIVE, Score: 0.9996

Sentence: 'The plot was a bit confusing, but the visuals were stunning.'
Prediction: POSITIVE, Score: 0.9998



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [17]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This movie was an absolute masterpiece; I loved every minute of it!"
prediction = sentiment_pipeline(sentence)
prediction

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998568296432495}]

In [19]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.label_map = self.model.config.id2label

    def recognize(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)

        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        predicted_labels = [self.label_map[p.item()] for p in predictions[0]]

        entities = []
        current_entity = None
        current_entity_tokens = []
        current_entity_start = -1

        for i, (token, label) in enumerate(zip(tokens, predicted_labels)):
            if token in ["[CLS]", "[SEP]", "[PAD]"]:
                if current_entity:
                    full_text = self.tokenizer.convert_tokens_to_string(current_entity_tokens)
                    entities.append({"text": full_text.replace(' ##', ''), "entity": current_entity.replace('B-','').replace('I-',''), "start": current_entity_start, "end": i-1})
                current_entity = None
                current_entity_tokens = []
                current_entity_start = -1
                continue

            if label.startswith("B-"):
                if current_entity:
                    full_text = self.tokenizer.convert_tokens_to_string(current_entity_tokens)
                    entities.append({"text": full_text.replace(' ##', ''), "entity": current_entity.replace('B-','').replace('I-',''), "start": current_entity_start, "end": i-1})
                current_entity = label
                current_entity_tokens = [token]
                current_entity_start = i
            elif label.startswith("I-"):
                if current_entity and current_entity.endswith(label[1:]): # Check if I- tag matches previous B- or I- tag type
                    current_entity_tokens.append(token)
                else: # If I- tag doesn't match, or no B-tag, treat as new entity or ignore.
                    if current_entity: # Close previous entity if any
                        full_text = self.tokenizer.convert_tokens_to_string(current_entity_tokens)
                        entities.append({"text": full_text.replace(' ##', ''), "entity": current_entity.replace('B-','').replace('I-',''), "start": current_entity_start, "end": i-1})
                    current_entity = None # Reset if mismatch or O-tag
                    current_entity_tokens = []
                    current_entity_start = -1
            else: # O-tag or other non-entity tag
                if current_entity:
                    full_text = self.tokenizer.convert_tokens_to_string(current_entity_tokens)
                    entities.append({"text": full_text.replace(' ##', ''), "entity": current_entity.replace('B-','').replace('I-',''), "start": current_entity_start, "end": i-1})
                current_entity = None
                current_entity_tokens = []
                current_entity_start = -1

        # Add any remaining entity after loop
        if current_entity:
            full_text = self.tokenizer.convert_tokens_to_string(current_entity_tokens)
            entities.append({"text": full_text.replace(' ##', ''), "entity": current_entity.replace('B-','').replace('I-',''), "start": current_entity_start, "end": len(tokens)-1})

        # Adjust start and end to be token indices relative to original sentence
        # This is a simplification; for exact character offsets, one would need to map token indices back to original string positions.
        # For this exercise, token indices are sufficient.
        return entities


In [20]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University in California."
recognized_entities = ner.recognize(sample_text)
for entity in recognized_entities:
    print(entity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'text': 'Google', 'entity': 'ORG', 'start': 1, 'end': 1}
{'text': 'Larry Page', 'entity': 'PER', 'start': 5, 'end': 6}
{'text': 'Sergey Brin', 'entity': 'PER', 'start': 8, 'end': 10}
{'text': 'Stanford University', 'entity': 'ORG', 'start': 20, 'end': 21}
{'text': 'California', 'entity': 'LOC', 'start': 23, 'end': 23}


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only Transformer | Decoder-only Transformer |
| Primary purpose | Understanding (bidirectional context) | Generation (unidirectional context) |
| Typical use cases | Sentiment analysis, NER, Q&A, text classification | Text generation, summarization, translation, chatbots |
| Strengths | Excellent for understanding nuances of context; good for classification | Highly generative and creative; good for open-ended tasks |
| Weaknesses | Not designed for direct text generation; primarily for discriminative tasks | Can struggle with factual accuracy; limited by context window for long texts |

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:

1.  **Describe how BERT encodes queries and documents.**
    BERT (Bidirectional Encoder Representations from Transformers) encodes both the user's query and the documents (or chunks of documents) from a knowledge base into high-dimensional numerical vectors called embeddings. It processes text bidirectionally, meaning it considers the context from both left and right of a word simultaneously. These embeddings capture the semantic meaning of the text, such that semantically similar pieces of text will have embeddings that are numerically close to each other in the vector space.

2.  **Explain how those embeddings are stored and searched in a vector database.**
    Once BERT generates embeddings for all documents in the knowledge base, these embeddings are stored in a specialized vector database (e.g., Pinecone, Weaviate, Milvus). Along with the embeddings, metadata or references to the original text chunks are also stored. When a user submits a query, BERT first converts this query into an embedding. The vector database then performs a similarity search (e.g., using cosine similarity or Euclidean distance) to find document embeddings that are most similar to the query embedding. This process efficiently retrieves the most semantically relevant documents or passages.

3.  **Outline how the retrieved passages are handed to a generative model like GPT.**
    After the most relevant passages are retrieved from the vector database, they are then passed as context to a generative large language model (LLM), such as GPT. The LLM receives the original user query along with these retrieved passages. The prompt to the LLM is typically constructed by concatenating the user's query with the relevant passages, guiding the LLM to generate a response that is grounded in the provided factual information, thereby reducing hallucinations and improving accuracy.

4.  **Provide a concrete application example (industry or product) where RAG with BERT makes sense.**
    A concrete application example where RAG with BERT makes sense is in **customer support chatbots for complex products or services**. Imagine a user asking a chatbot a highly specific question about a product's troubleshooting steps. Instead of relying solely on the LLM's pre-trained knowledge (which might be outdated or insufficient), a RAG system first uses BERT to embed the user's question and retrieve the most relevant articles or manuals from the company's extensive documentation knowledge base. These retrieved passages, containing the exact troubleshooting steps, are then fed to a generative LLM (like GPT) which synthesizes a precise and accurate answer, directly citing or summarizing information from the company's official sources. This ensures that customers receive consistent, factual, and up-to-date information.